# Cài thư viện

In [1]:
!pip install -q torchinfo

import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.utils.class_weight import compute_class_weight
from torchinfo import summary

from tqdm import tqdm
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Mô hình

In [2]:
class ResidualBlock1D(nn.Module):
    def __init__(self, channels, kernel_size=5):
        super().__init__()
        padding = kernel_size // 2

        self.conv1 = nn.Conv1d(channels, channels, kernel_size, padding=padding)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size, padding=padding)
        self.conv3 = nn.Conv1d(channels, channels, kernel_size, padding=padding)

        self.relu = nn.ReLU(inplace=True)
        self.pool = nn.MaxPool1d(2)

    def forward(self, x):
        identity = self.pool(x)   # downsample skip path

        out = self.conv1(x)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.relu(out)
        out = self.conv3(out)

        out = self.pool(out)
        out = out + identity
        out = self.relu(out)

        return out


In [3]:
class ECG_CNN(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()

        # Input: (B, 1, 187)
        self.stem = nn.Conv1d(1, 32, kernel_size=5, padding=2)

        # 5 Residual Blocks
        self.res_blocks = nn.Sequential(
            ResidualBlock1D(32),
            ResidualBlock1D(32),
            ResidualBlock1D(32),
            ResidualBlock1D(32),
            ResidualBlock1D(32),
        )

        # Sau 5 lần pool: 187 → ~5
        self.fc1 = nn.Linear(32 * 5, 128)
        self.fc2 = nn.Linear(128, num_classes)

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        # x: (B, 187)
        x = x.unsqueeze(1)          # (B, 1, 187)
        x = self.stem(x)            # (B, 32, 187)
        x = self.res_blocks(x)      # (B, 32, ~5)

        x = x.flatten(1)            # (B, 32*5)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)

        return x


# Test

In [4]:
# @torch.no_grad()
# def test(model,dataloader,device):
#     model.eval()

#     all_preds = []
#     all_labels = []

#     start_time = time.time()
#     total_samples = 0

#     for signals, labels in tqdm(dataloader, desc="Test", leave=False):
#         signals = signals.to(device)
#         labels  = labels.to(device)

#         logits = model(signals)            # (B, 5)
#         preds = torch.argmax(logits, dim=1)

#         all_preds.append(preds.cpu())
#         all_labels.append(labels.cpu())

#         total_samples += signals.size(0)

#     end_time = time.time()

#     all_preds = torch.cat(all_preds)
#     all_labels = torch.cat(all_labels)

#     results = {
#         "accuracy": accuracy_score(all_labels, all_preds),
#         "precision": precision_score(all_labels, all_preds, average="macro", zero_division=0),
#         "recall": recall_score(all_labels, all_preds, average="macro", zero_division=0),
#         "inference_time": (end_time - start_time) / total_samples
#     }

#     return results


# test_metrics = test(
#     model,
#     test_loader,
#     device
# )

# print(
#     f"TEST RESULTS | "
#     f"Acc: {test_metrics['accuracy']:.4f} | "
#     f"Prec: {test_metrics['precision']:.4f} | "
#     f"Recall: {test_metrics['recall']:.4f} | "
#     f"Infer: {test_metrics['inference_time']*1000:.2f} ms/sample"
# )

NameError: name 'model' is not defined

# Tiền xử lý

In [27]:
import numpy as np
import torch
from scipy.signal import butter, filtfilt, find_peaks
from scipy.interpolate import interp1d


# ===== 1. Bandpass Filter =====
def bandpass_filter(signal, fs=125, lowcut=0.5, highcut=40, order=3):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    filtered = filtfilt(b, a, signal)
    return filtered


# ===== 2. Normalize =====
def normalize_signal(signal):
    mean = np.mean(signal)
    std = np.std(signal)
    return (signal - mean) / (std + 1e-8)


# ===== 3. Resize về 187 samples =====
def resize_beat(beat, target_len=187):
    x_old = np.linspace(0, 1, len(beat))
    x_new = np.linspace(0, 1, target_len)
    f = interp1d(x_old, beat, kind='linear')
    return f(x_new)


# ===== 4. Full preprocessing pipeline =====
def preprocess_ecg_signal(raw_signal, fs=125):

    # Filter
    filtered = bandpass_filter(raw_signal, fs)

    # Detect R-peaks
    peaks, _ = find_peaks(filtered, distance=fs*0.4)

    beats = []

    for peak in peaks:
        start = peak - int(0.3 * fs)
        end   = peak + int(0.4 * fs)

        if start >= 0 and end < len(filtered):
            beat = filtered[start:end]
            beat = resize_beat(beat, 187)
            beat = normalize_signal(beat)
            beats.append(beat)

    beats = np.array(beats)

    # Convert sang tensor (N, 1, 187)
    beats_tensor = torch.tensor(beats, dtype=torch.float32)

    return beats_tensor


# Tạo tín hiệu giả lập

In [28]:
def simulate_ecg(duration=10, fs=125):
    t = np.linspace(0, duration, duration*fs)

    # Tạo sóng nhịp tim cơ bản
    heart_rate = 75  # bpm
    rr_interval = 60 / heart_rate

    signal = np.zeros_like(t)

    for beat_time in np.arange(0, duration, rr_interval):
        peak_index = int(beat_time * fs)
        if peak_index < len(signal):
            signal[peak_index] = 1.0

    # Làm mượt để giống ECG hơn
    signal = np.convolve(signal, np.hanning(10), mode='same')

    # Thêm nhiễu nhẹ
    noise = np.random.normal(0, 0.05, len(signal))
    signal = signal + noise

    return signal

In [29]:
# Giả lập dữ liệu từ máy đo
raw_ecg = simulate_ecg(duration=10, fs=125)

# Tiền xử lý
beats_tensor = preprocess_ecg_signal(raw_ecg, fs=125).to(device)

print("Shape đưa vào model:", beats_tensor.shape)

Shape đưa vào model: torch.Size([14, 187])


In [30]:
model = ECG_CNN(num_classes=5).to(device)
ckpt = torch.load("/content/drive/MyDrive/Colab Notebooks/CNM/project/best_epoch_4_acc_0.1791.pth", map_location=device)

model.load_state_dict(ckpt["model_state"])

<All keys matched successfully>

In [31]:
model.eval()
with torch.no_grad():
    logits = model(beats_tensor)
    preds = torch.argmax(logits, dim=1)

print("Predictions:", preds)

Predictions: tensor([2, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0])


In [26]:
class_names = {
    0: "N - Normal",
    1: "S - Supraventricular",
    2: "V - Ventricular",
    3: "F - Fusion",
    4: "Q - Unknown"
}